# 0-14 Yaş Oranı ve Hastane Sayısı Tahmin Notebook'u

Bu notebook iki hedef için temiz bir deney akışı kurar:

1. `yas_0_14`: İl bazlı 0-14 yaş nüfus oranı
2. `hastane_sayisi`: İl bazlı hastane sayısı

Özellikle önceki tabloda görülen **Accuracy yüksek ama R² negatif** problemini teşhis etmek için hedef değişken varyans kontrolleri, lag baseline ve zaman sıralı fold değerlendirmesi içerir.

In [46]:
# ============================================================
# 1) KÜTÜPHANELER
# ============================================================

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    BaggingRegressor,
    StackingRegressor,
    VotingRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except Exception:
    HAS_CAT = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [48]:
# ============================================================
# 2) DOSYA YOLLARI
# ============================================================

MODEL_GIRIS_PATH = "model_giris.csv"
YAS_PATH = "2025nufus-yas.csv"

print("model_giris var mı?", os.path.exists(MODEL_GIRIS_PATH))
print("yaş dosyası var mı?", os.path.exists(YAS_PATH))

model_giris var mı? True
yaş dosyası var mı? True


In [50]:
# ============================================================
# 3) YARDIMCI FONKSİYONLAR
# ============================================================

TR_MAP = str.maketrans({
    "İ": "i", "I": "i", "ı": "i",
    "Ş": "s", "ş": "s",
    "Ğ": "g", "ğ": "g",
    "Ü": "u", "ü": "u",
    "Ö": "o", "ö": "o",
    "Ç": "c", "ç": "c",
})

def normalize_city_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"-\d+$", "", x)
    x = x.translate(TR_MAP).lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def clean_numeric(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip().replace(",", ".")
    s = re.sub(r"[^0-9.\-]", "", s)
    if s in ["", "-", ".", "-."]:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def calc_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    if len(np.unique(y_true)) <= 1:
        r2 = np.nan
    else:
        r2 = r2_score(y_true, y_pred)

    eps = 1e-9
    mape = np.mean(
        np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))
    ) * 100

    accuracy = max(0, 100 - mape)

    return {
        "Accuracy": accuracy,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "ErrorRate": mape
    }

def show_target_diagnostics(df, target):
    print("\n" + "="*80)
    print(f"HEDEF KONTROLÜ: {target}")
    print("="*80)

    print("\nGenel özet:")
    print(df[target].describe())

    print("\nYıl bazında varyans kontrolü:")
    diag = (
        df.groupby("yıl")[target]
        .agg(["count", "nunique", "mean", "std", "min", "max"])
        .round(4)
    )
    display(diag.tail(15))

    low_var = diag[(diag["std"].fillna(0) < 1e-6) | (diag["nunique"] <= 1)]

    if len(low_var) > 0:
        print("\nUYARI: Bazı yıllarda hedef değişken tek değerli/çok düşük varyanslı görünüyor.")
        display(low_var)
    else:
        print("\nSorun yok: Yıl bazında hedef değişkende şehirler arası varyans var.")

## 4) Ana veri setini oku

`model_giris.csv` içinde `il_adı`, `yıl`, `hastane`, `nufus` sütunları beklenir.

In [53]:
# ============================================================
# 4) MODEL GİRİŞ VERİSİNİ OKU
# ============================================================

df_main = pd.read_csv(MODEL_GIRIS_PATH)
df_main.columns = [c.strip() for c in df_main.columns]

required_cols = ["il_adı", "yıl", "hastane", "nufus"]
missing = [c for c in required_cols if c not in df_main.columns]
if missing:
    raise ValueError(f"Eksik kolonlar: {missing}")

df_main["il_key"] = df_main["il_adı"].apply(normalize_city_name)
df_main["yıl"] = pd.to_numeric(df_main["yıl"], errors="coerce").astype("Int64")

numeric_cols = [c for c in df_main.columns if c not in ["il_adı", "il_key"]]
for c in numeric_cols:
    df_main[c] = pd.to_numeric(df_main[c], errors="coerce")

df_main = df_main.dropna(subset=["il_key", "yıl", "nufus", "hastane"]).copy()
df_main["yıl"] = df_main["yıl"].astype(int)

print(df_main.shape)
display(df_main.head())
print("Yıl aralığı:", df_main["yıl"].min(), "-", df_main["yıl"].max())
print("İl sayısı:", df_main["il_adı"].nunique())

(1863, 12)


,il_adı,yıl,hastane,yatak,nufus,asistan,diger,ebe,hemsire,pratisyen,uzman,il_key
0,Adana,2002,18.0,4266.0,1916637.0,646.0,1182.0,1110.0,1927.0,826.0,1227.0,adana
1,Adana,2003,21.0,4382.0,1933428.0,794.0,1414.0,1147.0,2118.0,844.0,1439.0,adana
2,Adana,2004,21.0,4207.0,1951142.0,856.0,1469.0,1167.0,2157.0,892.0,1497.0,adana
3,Adana,2005,22.0,4773.0,1969512.0,869.0,1540.0,1186.0,2223.0,979.0,1555.0,adana
4,Adana,2006,21.0,4973.0,1988277.0,409.0,1683.0,1258.0,2235.0,938.0,1364.0,adana


Yıl aralığı: 2002 - 2024
İl sayısı: 81


## 5) Yaş dosyasını oku ve `yas_0_14` hedefini oluştur

Yaş dosyası `|` ayracıyla özel formatta geldiği için özel parser kullanılır.

Formül:

```python
yas_0_14 = (yas_0_4 + yas_5_9 + yas_10_14) / nufus * 100
```

In [58]:
# ============================================================
# 5) YAŞ DOSYASINI PARSE ET
# ============================================================

def parse_age_file(path):
    rows = []
    current_year = None

    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split("|")
            first = parts[0].strip() if len(parts) > 0 else ""
            second = parts[1].strip() if len(parts) > 1 else ""

            if first.isdigit():
                current_year = int(first)
                city_raw = second
                values = parts[2:]
            elif current_year is not None and second:
                city_raw = second
                values = parts[2:]
            else:
                continue

            if len(values) < 9:
                continue

            rows.append({
                "yıl": current_year,
                "il_raw": city_raw,
                "il_key": normalize_city_name(city_raw),
                "yas_0_4": clean_numeric(values[0]),
                "yas_10_14": clean_numeric(values[1]),
                "yas_5_9": clean_numeric(values[2]),
                "yas_65_69": clean_numeric(values[3]),
                "yas_70_74": clean_numeric(values[4]),
                "yas_75_79": clean_numeric(values[5]),
                "yas_80_84": clean_numeric(values[6]),
                "yas_85_89": clean_numeric(values[7]),
                "yas_90_plus": clean_numeric(values[8]),
            })

    age = pd.DataFrame(rows)

    if age.empty:
        raise ValueError(
            "Yaş dosyası parse edilemedi. Dosyanın | ayraçlı olup olmadığını ve sütun sırasını kontrol et."
        )

    age = age.dropna(subset=["il_key", "yıl"]).copy()
    age["yıl"] = age["yıl"].astype(int)

    return age


df_age = parse_age_file(YAS_PATH)

print(df_age.shape)
display(df_age.head())
print("Yaş dosyası yıl aralığı:", df_age["yıl"].min(), "-", df_age["yıl"].max())
print("Yaş dosyası il sayısı:", df_age["il_key"].nunique())

(1539, 12)


,yıl,il_raw,il_key,yas_0_4,yas_10_14,yas_5_9,yas_65_69,yas_70_74,yas_75_79,yas_80_84,yas_85_89,yas_90_plus
0,2007,Adana-1,adana,170903.0,196529.0,192331.0,40794.0,30617.0,23068.0,12182.0,4004.0,2430.0
1,2007,Adıyaman-2,adiyaman,60457.0,69026.0,65992.0,11596.0,8838.0,7487.0,3828.0,1449.0,838.0
2,2007,Afyonkarahisar-3,afyonkarahisar,55658.0,61309.0,61660.0,21446.0,18284.0,14187.0,6239.0,1547.0,691.0
3,2007,Ağrı-4,agri,66964.0,70488.0,82191.0,6722.0,5069.0,3804.0,2067.0,827.0,318.0
4,2007,Aksaray-68,aksaray,32596.0,36850.0,37386.0,8575.0,6885.0,4498.0,2465.0,841.0,441.0


Yaş dosyası yıl aralığı: 2007 - 2025
Yaş dosyası il sayısı: 81


In [62]:
# ============================================================
# 6) ANA VERİ + YAŞ VERİSİ MERGE
# ============================================================

df = df_main.merge(
    df_age[["il_key", "yıl", "yas_0_4", "yas_5_9", "yas_10_14"]],
    on=["il_key", "yıl"],
    how="left"
)

df["yas_0_14"] = (
    df["yas_0_4"] + df["yas_5_9"] + df["yas_10_14"]
) / df["nufus"] * 100

df["hastane_sayisi"] = df["hastane"]

print(df.shape)
display(
    df[[
        "il_adı", "yıl", "nufus",
        "yas_0_4", "yas_5_9", "yas_10_14",
        "yas_0_14", "hastane_sayisi"
    ]].head(10)
)

print("\nyas_0_14 boş sayısı:", df["yas_0_14"].isna().sum())
print("hastane_sayisi boş sayısı:", df["hastane_sayisi"].isna().sum())

(1863, 17)


,il_adı,yıl,nufus,yas_0_4,yas_5_9,yas_10_14,yas_0_14,hastane_sayisi
0,Adana,2002,1916637.0,NaN,NaN,NaN,NaN,18.0
1,Adana,2003,1933428.0,NaN,NaN,NaN,NaN,21.0
2,Adana,2004,1951142.0,NaN,NaN,NaN,NaN,21.0
3,Adana,2005,1969512.0,NaN,NaN,NaN,NaN,22.0
4,Adana,2006,1988277.0,NaN,NaN,NaN,NaN,21.0
5,Adana,2007,2006650.0,170903.0,192331.0,196529.0,27.895398,20.0
6,Adana,2008,2026319.0,179673.0,187993.0,196006.0,27.817535,21.0
7,Adana,2009,2062226.0,187143.0,183958.0,195251.0,27.463139,24.0
8,Adana,2010,2085225.0,186862.0,182019.0,197410.0,27.157309,25.0
9,Adana,2011,2108805.0,186032.0,181099.0,198518.0,26.823201,25.0



yas_0_14 boş sayısı: 405
hastane_sayisi boş sayısı: 0


## 6.5) Hedef değişken teşhisi

Burada `nunique`, `std`, `min`, `max` değerlerine bak.

Eğer bir yılda `nunique = 1` veya `std = 0` ise hedef değişken şehir bazlı değildir veya yanlış merge edilmiştir.

In [65]:
# ============================================================
# 6.5) TARGET DIAGNOSTICS
# ============================================================

show_target_diagnostics(df.dropna(subset=["yas_0_14"]), "yas_0_14")
show_target_diagnostics(df.dropna(subset=["hastane_sayisi"]), "hastane_sayisi")


HEDEF KONTROLÜ: yas_0_14

Genel özet:
count    1458.000000
mean       23.774550
std         6.575846
min        13.926477
25%        19.045591
50%        21.857943
75%        26.933072
max        44.982825
Name: yas_0_14, dtype: float64

Yıl bazında varyans kontrolü:


,count,nunique,mean,std,min,max
yıl,,,,,,
2010,81,81,25.8949,7.0992,16.6671,44.8716
2011,81,81,25.4047,6.7563,15.0525,42.4929
2012,81,81,24.9456,6.7099,14.5347,41.9740
2013,81,81,24.5064,6.6704,14.9424,41.4864
2014,81,81,24.0612,6.5390,14.7942,40.7747
2015,81,81,23.7314,6.3901,14.5058,40.4047
2016,81,81,23.4034,6.2844,14.5061,40.1133
2017,81,81,23.1780,6.1581,14.3931,39.9902
2018,81,81,22.8619,6.0310,14.1420,39.8274



Sorun yok: Yıl bazında hedef değişkende şehirler arası varyans var.

HEDEF KONTROLÜ: hastane_sayisi

Genel özet:
count    1863.000000
mean       17.252281
std        25.455901
min         1.000000
25%         7.000000
50%        12.000000
75%        20.000000
max       238.000000
Name: hastane_sayisi, dtype: float64

Yıl bazında varyans kontrolü:


,count,nunique,mean,std,min,max
yıl,,,,,,
2010,81,29,17.2099,25.0199,1.0,216.0
2011,81,33,17.3827,24.8810,1.0,213.0
2012,81,34,17.7531,25.6289,1.0,219.0
2013,81,32,18.1852,26.7834,1.0,229.0
2014,81,32,18.4074,27.2070,1.0,232.0
2015,81,31,18.4815,27.1248,1.0,232.0
2016,81,34,18.6420,27.8156,1.0,238.0
2017,81,35,18.7407,27.9485,1.0,238.0
2018,81,36,18.9383,27.8215,1.0,236.0



Sorun yok: Yıl bazında hedef değişkende şehirler arası varyans var.


## 7) Feature engineering

Her hedef için ayrı lag özellikleri üretilir:

- `target_lag1`
- `target_lag2`
- `target_lag3`
- `target_roll3`
- `target_diff1`

Bu yapı leakage riskini azaltır.

In [68]:
# ============================================================
# 7) FEATURE ENGINEERING
# ============================================================

def build_target_dataset(df, target):
    use_cols = ["il_adı", "il_key", "yıl", "nufus", target]
    d = df[use_cols].dropna(subset=[target, "nufus", "yıl", "il_key"]).copy()
    d = d.sort_values(["il_key", "yıl"]).reset_index(drop=True)

    d["log_nufus"] = np.log1p(d["nufus"])
    d["city_time_index"] = d.groupby("il_key").cumcount()

    d[f"{target}_lag1"] = d.groupby("il_key")[target].shift(1)
    d[f"{target}_lag2"] = d.groupby("il_key")[target].shift(2)
    d[f"{target}_lag3"] = d.groupby("il_key")[target].shift(3)
    d[f"{target}_roll3"] = (
        d.groupby("il_key")[target]
        .shift(1)
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )
    d[f"{target}_diff1"] = d[f"{target}_lag1"] - d[f"{target}_lag2"]

    d = d.dropna(subset=[f"{target}_lag1", f"{target}_lag2"]).copy()
    return d

data_yas = build_target_dataset(df, "yas_0_14")
data_hastane = build_target_dataset(df, "hastane_sayisi")

print("yas_0_14 model datası:", data_yas.shape, data_yas["yıl"].min(), "-", data_yas["yıl"].max())
print("hastane_sayisi model datası:", data_hastane.shape, data_hastane["yıl"].min(), "-", data_hastane["yıl"].max())

display(data_yas.head())
display(data_hastane.head())

yas_0_14 model datası: (1296, 12) 2009 - 2024
hastane_sayisi model datası: (1701, 12) 2004 - 2024


,il_adı,il_key,yıl,nufus,yas_0_14,log_nufus,city_time_index,yas_0_14_lag1,yas_0_14_lag2,yas_0_14_lag3,yas_0_14_roll3,yas_0_14_diff1
2,Adana,adana,2009,2062226.0,27.463139,14.539297,2,27.817535,27.895398,NaN,27.856466,-0.077863
3,Adana,adana,2010,2085225.0,27.157309,14.550388,3,27.463139,27.817535,27.895398,27.725357,-0.354396
4,Adana,adana,2011,2108805.0,26.823201,14.561632,4,27.157309,27.463139,27.817535,27.479328,-0.305830
5,Adana,adana,2012,2125635.0,26.565003,14.569582,5,26.823201,27.157309,27.463139,27.147883,-0.334108
6,Adana,adana,2013,2149260.0,26.255223,14.580635,6,26.565003,26.823201,27.157309,26.848504,-0.258198


,il_adı,il_key,yıl,nufus,hastane_sayisi,log_nufus,city_time_index,hastane_sayisi_lag1,hastane_sayisi_lag2,hastane_sayisi_lag3,hastane_sayisi_roll3,hastane_sayisi_diff1
2,Adana,adana,2004,1951142.0,21.0,14.483926,2,21.0,18.0,NaN,19.500000,3.0
3,Adana,adana,2005,1969512.0,22.0,14.493297,3,21.0,21.0,18.0,20.000000,0.0
4,Adana,adana,2006,1988277.0,21.0,14.502779,4,22.0,21.0,21.0,21.333333,1.0
5,Adana,adana,2007,2006650.0,20.0,14.511978,5,21.0,22.0,21.0,21.333333,-1.0
6,Adana,adana,2008,2026319.0,21.0,14.521732,6,20.0,21.0,22.0,21.000000,-1.0


## 8) Model listesi

XGBoost/CatBoost kuruluysa otomatik eklenir.

In [71]:
# ============================================================
# 8) MODEL TANIMLARI
# ============================================================

def get_models():
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0),
        "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=10000, random_state=42),
        "KNN": KNeighborsRegressor(n_neighbors=5),
        "SVR": SVR(C=10, epsilon=0.1),
        "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, min_samples_leaf=2),
        "ExtraTrees": ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1, min_samples_leaf=2),
        "GradientBoosting": GradientBoostingRegressor(random_state=42),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42),
        "Bagging": BaggingRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    }

    if HAS_XGB:
        models["XGBoost"] = XGBRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1
        )

    if HAS_CAT:
        models["CatBoost"] = CatBoostRegressor(
            iterations=300,
            learning_rate=0.03,
            depth=4,
            loss_function="RMSE",
            random_seed=42,
            verbose=False
        )

    base_for_ensemble = [
        ("ridge", Ridge(alpha=1.0)),
        ("rf", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1, min_samples_leaf=2)),
        ("gb", GradientBoostingRegressor(random_state=42)),
    ]

    models["Voting"] = VotingRegressor(base_for_ensemble)
    models["Stacking"] = StackingRegressor(
        estimators=base_for_ensemble,
        final_estimator=Ridge(alpha=1.0),
        passthrough=False
    )

    return models

models = get_models()
print("Model sayısı:", len(models))
print(list(models.keys()))

Model sayısı: 14
['LinearRegression', 'Ridge', 'ElasticNet', 'KNN', 'SVR', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'HistGradientBoosting', 'Bagging', 'XGBoost', 'CatBoost', 'Voting', 'Stacking']


## 9) Zaman sıralı fold değerlendirme

Random split yoktur. Her fold'da geçmiş yıllar train, tek gelecek yıl test olur.

Örnek:

```text
Train: 2007-2019 → Test: 2020
Train: 2007-2020 → Test: 2021
Train: 2007-2021 → Test: 2022
Train: 2007-2022 → Test: 2023
Train: 2007-2023 → Test: 2024
```

In [74]:
# ============================================================
# 9) TIME-BASED CV
# ============================================================

def evaluate_target(data, target, test_years=None, min_train_years=8):
    d = data.copy().sort_values(["yıl", "il_key"]).reset_index(drop=True)

    if test_years is None:
        years = sorted(d["yıl"].unique())
        test_years = years[-5:]

    categorical_features = ["il_adı"]
    numeric_features = [
        "yıl",
        "nufus",
        "log_nufus",
        "city_time_index",
        f"{target}_lag1",
        f"{target}_lag2",
        f"{target}_lag3",
        f"{target}_roll3",
        f"{target}_diff1",
    ]

    feature_cols = categorical_features + numeric_features

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_features),
        ],
        remainder="drop"
    )

    results = []
    predictions = []

    for test_year in test_years:
        train = d[d["yıl"] < test_year].copy()
        test = d[d["yıl"] == test_year].copy()

        if train["yıl"].nunique() < min_train_years or len(test) == 0:
            continue

        X_train = train[feature_cols]
        y_train = train[target]
        X_test = test[feature_cols]
        y_test = test[target]

        baseline_pred = test[f"{target}_lag1"].values
        base_metrics = calc_regression_metrics(y_test, baseline_pred)
        results.append({
            "Hedef": target,
            "Model": "Baseline_Lag1",
            "Fold_Test_Year": test_year,
            **base_metrics
        })

        tmp_pred = test[["il_adı", "il_key", "yıl"]].copy()
        tmp_pred["Hedef"] = target
        tmp_pred["Model"] = "Baseline_Lag1"
        tmp_pred["y_true"] = y_test.values
        tmp_pred["y_pred"] = baseline_pred
        predictions.append(tmp_pred)

        for model_name, model in get_models().items():
            pipe = Pipeline(steps=[
                ("preprocess", preprocessor),
                ("model", clone(model))
            ])

            try:
                pipe.fit(X_train, y_train)
                pred = pipe.predict(X_test)
                metrics = calc_regression_metrics(y_test, pred)

                results.append({
                    "Hedef": target,
                    "Model": model_name,
                    "Fold_Test_Year": test_year,
                    **metrics
                })

                tmp_pred = test[["il_adı", "il_key", "yıl"]].copy()
                tmp_pred["Hedef"] = target
                tmp_pred["Model"] = model_name
                tmp_pred["y_true"] = y_test.values
                tmp_pred["y_pred"] = pred
                predictions.append(tmp_pred)

            except Exception as e:
                results.append({
                    "Hedef": target,
                    "Model": model_name,
                    "Fold_Test_Year": test_year,
                    "Accuracy": np.nan,
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "R2": np.nan,
                    "ErrorRate": np.nan,
                    "Error": str(e)
                })

    result_df = pd.DataFrame(results)
    pred_df = pd.concat(predictions, ignore_index=True) if predictions else pd.DataFrame()
    return result_df, pred_df

res_yas, pred_yas = evaluate_target(data_yas, "yas_0_14")
res_hastane, pred_hastane = evaluate_target(data_hastane, "hastane_sayisi")

results_all = pd.concat([res_yas, res_hastane], ignore_index=True)
preds_all = pd.concat([pred_yas, pred_hastane], ignore_index=True)

display(results_all.head())
print(results_all.shape)

,Hedef,Model,Fold_Test_Year,Accuracy,MAE,RMSE,R2,ErrorRate,Error
0,yas_0_14,Baseline_Lag1,2020,98.731917,0.304050,0.366824,0.995900,1.268083,NaN
1,yas_0_14,LinearRegression,2020,99.058364,0.199329,0.252986,0.998050,0.941636,NaN
2,yas_0_14,Ridge,2020,99.018921,0.208609,0.253371,0.998044,0.981079,NaN
3,yas_0_14,ElasticNet,2020,99.109393,0.183127,0.228971,0.998403,0.890607,NaN
4,yas_0_14,KNN,2020,98.120475,0.418328,0.532557,0.991358,1.879525,NaN


(150, 9)


## 10) Fold ortalamaları ve en iyi model seçimi

Önceki problemde model seçimi sadece `Accuracy` ile yapıldığı için R² kötü olan model seçilebiliyordu.

Burada sıralama:

1. R² yüksek
2. RMSE düşük
3. MAE düşük
4. ErrorRate düşük

In [81]:
# ============================================================
# 10) ÖZET TABLOLAR
# ============================================================

summary = (
    results_all
    .groupby(["Hedef", "Model"], as_index=False)
    .agg({
        "Accuracy": "mean",
        "MAE": "mean",
        "RMSE": "mean",
        "R2": "mean",
        "ErrorRate": "mean"
    })
)

summary = summary.sort_values(
    ["Hedef", "R2", "RMSE", "MAE"],
    ascending=[True, False, True, True]
).reset_index(drop=True)

print("========== HEDEF × MODEL ORTALAMA SONUÇLAR ==========")
display(summary.round(4))

best_models = (
    summary
    .sort_values(
        ["Hedef", "R2", "RMSE", "MAE", "ErrorRate"],
        ascending=[True, False, True, True, True]
    )
    .groupby("Hedef", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("\n========== HER HEDEF İÇİN EN İYİ MODEL ==========")
display(best_models.round(4))

========== HEDEF × MODEL ORTALAMA SONUÇLAR ==========


,Hedef,Model,Accuracy,MAE,RMSE,R2,ErrorRate
0,hastane_sayisi,Baseline_Lag1,98.4401,0.2519,0.6162,0.9995,1.5599
1,hastane_sayisi,XGBoost,96.5714,0.4474,0.7060,0.9993,3.4286
2,hastane_sayisi,Voting,97.3619,0.3918,0.7060,0.9993,2.6381
3,hastane_sayisi,GradientBoosting,96.7282,0.4417,0.7245,0.9993,3.2718
4,hastane_sayisi,Bagging,97.3085,0.3978,0.7162,0.9993,2.6915
5,hastane_sayisi,RandomForest,97.2189,0.4031,0.7192,0.9993,2.7811
6,hastane_sayisi,ExtraTrees,97.4161,0.3975,0.7342,0.9993,2.5839
7,hastane_sayisi,Stacking,97.0064,0.4310,0.7690,0.9992,2.9936
8,hastane_sayisi,LinearRegression,96.6638,0.4603,0.7705,0.9992,3.3362
9,hastane_sayisi,Ridge,97.2252,0.4437,0.8603,0.9990,2.7748



========== HER HEDEF İÇİN EN İYİ MODEL ==========


,Hedef,Model,Accuracy,MAE,RMSE,R2,ErrorRate
0,hastane_sayisi,Baseline_Lag1,98.4401,0.2519,0.6162,0.9995,1.5599
1,yas_0_14,Voting,99.1310,0.1799,0.2297,0.9982,0.8690


## 11) En iyi model tahminlerini incele

In [84]:
# ============================================================
# 11) EN İYİ MODEL TAHMİNLERİNİ İNCELE
# ============================================================

for _, row in best_models.iterrows():
    target = row["Hedef"]
    model = row["Model"]

    print("\n" + "="*100)
    print(f"HEDEF: {target} | EN İYİ MODEL: {model}")
    print("="*100)

    pp = preds_all[
        (preds_all["Hedef"] == target) & 
        (preds_all["Model"] == model)
    ].copy()

    pp["abs_error"] = np.abs(pp["y_true"] - pp["y_pred"])
    pp["pct_error"] = (
        pp["abs_error"] / np.maximum(np.abs(pp["y_true"]), 1e-9) * 100
    )

    display(
        pp[["il_adı", "yıl", "y_true", "y_pred", "abs_error", "pct_error"]]
        .sort_values(["yıl", "abs_error"], ascending=[True, False])
        .head(20)
        .round(4)
    )

    print("Hata özeti:")
    display(pp[["abs_error", "pct_error"]].describe().round(4))


HEDEF: hastane_sayisi | EN İYİ MODEL: Baseline_Lag1


,il_adı,yıl,y_true,y_pred,abs_error,pct_error
5709,İstanbul,2020,232.0,235.0,3.0,1.2931
5698,Elazığ,2020,13.0,11.0,2.0,15.3846
5700,Erzurum,2020,21.0,23.0,2.0,9.5238
5673,Ağrı,2020,9.0,10.0,1.0,11.1111
5677,Antalya,2020,46.0,47.0,1.0,2.1739
5680,Aydın,2020,23.0,24.0,1.0,4.3478
5683,Batman,2020,13.0,12.0,1.0,7.6923
5689,Burdur,2020,8.0,9.0,1.0,12.5000
5695,Diyarbakır,2020,28.0,27.0,1.0,3.5714
5702,Gaziantep,2020,32.0,31.0,1.0,3.1250


Hata özeti:


,abs_error,pct_error
count,405.0000,405.0000
mean,0.2519,1.5599
std,0.5679,4.8669
min,0.0000,0.0000
25%,0.0000,0.0000
50%,0.0000,0.0000
75%,0.0000,0.0000
max,4.0000,66.6667



HEDEF: yas_0_14 | EN İYİ MODEL: Voting


,il_adı,yıl,y_true,y_pred,abs_error,pct_error
986,Bayburt,2020,20.1843,19.4114,0.7730,3.8297
1047,Tunceli,2020,14.9707,14.5056,0.4651,3.1067
985,Batman,2020,33.1292,33.5887,0.4595,1.3869
1014,Karabük,2020,16.3394,15.9345,0.4049,2.4781
1031,Muş,2020,33.8179,34.2201,0.4022,1.1893
1016,Kars,2020,26.5015,26.8855,0.3839,1.4487
1049,Van,2020,33.3084,33.6901,0.3817,1.1461
990,Bolu,2020,17.7331,17.3528,0.3802,2.1441
1001,Erzincan,2020,19.4113,19.0481,0.3632,1.8709
994,Çankırı,2020,17.8217,17.4622,0.3595,2.0172


Hata özeti:


,abs_error,pct_error
count,405.0000,405.0000
mean,0.1799,0.8690
std,0.1490,0.6997
min,0.0002,0.0007
25%,0.0795,0.3775
50%,0.1538,0.7262
75%,0.2440,1.1916
max,1.1763,3.9613


## 12) Son yıl hold-out testi

Son yıl, ayrıca raporlanabilir.

In [89]:
# ============================================================
# 12) SON YIL TEST SONUÇLARI
# ============================================================

last_year = int(df_main["yıl"].max())
last_year_results = results_all[results_all["Fold_Test_Year"] == last_year].copy()

last_year_summary = last_year_results.sort_values(
    ["Hedef", "R2", "RMSE", "MAE"],
    ascending=[True, False, True, True]
)

print(f"========== {last_year} TEST SONUÇLARI ==========")
display(last_year_summary.round(4))

best_2024 = last_year_summary.groupby("Hedef", as_index=False).head(1)

print(f"\n========== {last_year} İÇİN EN İYİLER ==========")
display(best_2024.round(4))

========== 2024 TEST SONUÇLARI ==========


,Hedef,Model,Fold_Test_Year,Accuracy,MAE,RMSE,R2,ErrorRate,Error
135,hastane_sayisi,Baseline_Lag1,2024,98.8076,0.2222,0.5879,0.9995,1.1924,NaN
136,hastane_sayisi,LinearRegression,2024,97.2110,0.4252,0.7152,0.9993,2.7890,NaN
146,hastane_sayisi,XGBoost,2024,96.9100,0.4336,0.7320,0.9993,3.0900,NaN
143,hastane_sayisi,GradientBoosting,2024,97.0374,0.4370,0.7660,0.9992,2.9626,NaN
148,hastane_sayisi,Voting,2024,97.4746,0.4173,0.7818,0.9992,2.5254,NaN
149,hastane_sayisi,Stacking,2024,97.2107,0.4514,0.8056,0.9991,2.7893,NaN
137,hastane_sayisi,Ridge,2024,97.3779,0.4443,0.8297,0.9991,2.6221,NaN
142,hastane_sayisi,ExtraTrees,2024,97.4452,0.4245,0.8593,0.9990,2.5548,NaN
141,hastane_sayisi,RandomForest,2024,97.1028,0.4577,0.8794,0.9989,2.8972,NaN
145,hastane_sayisi,Bagging,2024,97.1790,0.4693,0.8862,0.9989,2.8210,NaN



========== 2024 İÇİN EN İYİLER ==========


,Hedef,Model,Fold_Test_Year,Accuracy,MAE,RMSE,R2,ErrorRate,Error
135,hastane_sayisi,Baseline_Lag1,2024,98.8076,0.2222,0.5879,0.9995,1.1924,NaN
74,yas_0_14,Stacking,2024,99.3611,0.1305,0.1712,0.9990,0.6389,NaN


## 13) Final modeli tüm geçmiş veriyle eğit ve kaydet

In [94]:
# ============================================================
# 13) FINAL MODEL TRAINING
# ============================================================

import joblib
from sklearn.base import clone

def train_final_model(data, target, model_name):
    d = data.copy().sort_values(["yıl", "il_key"]).reset_index(drop=True)

    categorical_features = ["il_adı"]
    numeric_features = [
        "yıl",
        "nufus",
        "log_nufus",
        "city_time_index",
        f"{target}_lag1",
        f"{target}_lag2",
        f"{target}_lag3",
        f"{target}_roll3",
        f"{target}_diff1",
    ]

    feature_cols = categorical_features + numeric_features

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_features),
        ],
        remainder="drop"
    )

    models = get_models()

    if model_name not in models:
        raise ValueError(
            f"{model_name} get_models() içinde yok. "
            f"Mevcut modeller: {list(models.keys())}"
        )

    model = models[model_name]

    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", clone(model))
    ])

    X = d[feature_cols]
    y = d[target]

    pipe.fit(X, y)

    return pipe, feature_cols


final_models = {}

for _, row in best_models.iterrows():
    target = row["Hedef"]
    model_name = row["Model"]

    print("\n" + "="*80)
    print(f"Hedef: {target} | Seçilen model: {model_name}")
    print("="*80)

    if target == "yas_0_14":
        data_target = data_yas
    elif target == "hastane_sayisi":
        data_target = data_hastane
    else:
        print(f"{target} atlandı.")
        continue

    # Baseline gerçek bir sklearn modeli olmadığı için kaydedilmez
    if model_name == "Baseline_Lag1":
        print(
            f"{target} için en iyi sonuç Baseline_Lag1 çıkmış. "
            "Bu gerçek bir eğitilebilir model değil; lag1 tahmini kullanılıyor. "
            "Bu yüzden joblib modeli kaydedilmedi."
        )

        final_models[target] = {
            "model_name": model_name,
            "pipeline": None,
            "feature_cols": [f"{target}_lag1"],
            "note": "Baseline_Lag1: tahmin = son yıl değeri"
        }

        continue

    pipe, feature_cols = train_final_model(data_target, target, model_name)

    final_models[target] = {
        "model_name": model_name,
        "pipeline": pipe,
        "feature_cols": feature_cols
    }

    save_path = os.path.join(BASE_DIR, f"final_model_{target}_{model_name}.joblib")
    joblib.dump(final_models[target], save_path)

    print(f"Kaydedildi: {save_path}")


Hedef: hastane_sayisi | Seçilen model: Baseline_Lag1
hastane_sayisi için en iyi sonuç Baseline_Lag1 çıkmış. Bu gerçek bir eğitilebilir model değil; lag1 tahmini kullanılıyor. Bu yüzden joblib modeli kaydedilmedi.

Hedef: yas_0_14 | Seçilen model: Voting
Kaydedildi: /mnt/data\final_model_yas_0_14_Voting.joblib


## 14) Çıktıları CSV olarak kaydet

In [100]:
# ============================================================
# 14) ÇIKTILARI KAYDET
# ============================================================

summary_path = os.path.join("yas_0_14_hastane_model_ozet.csv")
fold_path = os.path.join("yas_0_14_hastane_fold_sonuclari.csv")
pred_path = os.path.join("yas_0_14_hastane_tahminler.csv")
best_path = os.path.join("yas_0_14_hastane_en_iyi_modeller.csv")

summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
results_all.to_csv(fold_path, index=False, encoding="utf-8-sig")
preds_all.to_csv(pred_path, index=False, encoding="utf-8-sig")
best_models.to_csv(best_path, index=False, encoding="utf-8-sig")

print("Kaydedilen dosyalar:")
print(summary_path)
print(fold_path)
print(pred_path)
print(best_path)

Kaydedilen dosyalar:
yas_0_14_hastane_model_ozet.csv
yas_0_14_hastane_fold_sonuclari.csv
yas_0_14_hastane_tahminler.csv
yas_0_14_hastane_en_iyi_modeller.csv


## 15) Yorumlama notu

Eğer yine `Accuracy yüksek ama R² negatif` sonucu görürsen, önce şunlara bak:

1. `show_target_diagnostics()` çıktısında yıl bazında `std` çok düşük mü?
2. Hedef değişken gerçekten il bazlı mı?
3. Model seçimi sadece `Accuracy` ile mi yapılıyor?
4. Baseline `Lag1`, ML modellerinden daha mı iyi?
5. Test yılındaki gerçek değer aralığı çok dar mı?

Özellikle `yas_0_14` için R² negatifse sorun çoğu zaman modelde değil, hedef değişkenin hazırlanma biçimindedir.